# Deep-learning surrogate models

Train a data-driven surrogate that maps a vector of design parameters to a discretised
response curve (e.g. 200+ output values) with one of six architectures:

| key | model | input layout |
|---|---|---|
| `ann` | feed-forward ANN | `(N, n_in)` |
| `cnn1d` | 1D CNN | `(N, n_in, 1)` |
| `rnn` | SimpleRNN | `(N, 1, n_in)` |
| `gru` | GRU | `(N, 1, n_in)` |
| `lstm` | LSTM (optionally bidirectional) | `(N, n_in, 1)` |
| `cnn_gru` | Conv1D front-end + GRU | `(N, n_in, 1)` |

All hyperparameters live in `configs/surrogate/<model>.yaml`; the same experiment can be run
from the command line with `surrogate-train configs/surrogate/lstm.yaml`.

**Pipeline** (`neural_pde_solvers.surrogate.run_experiment`)

1. Train / validation / test split — scalers fitted on the training block only.
2. Early stopping, LR-on-plateau and checkpointing on the **validation** block; the test
   block is used once, for the final report.
3. Metrics in scaled *and* physical units, artifacts saved to `output_dir`.

In [ ]:
# --- Setup -------------------------------------------------------------------
# Locally: `pip install -e ".[surrogate]"` from the repository root.
# On Colab the repository is cloned and installed automatically.
import os
import subprocess
import sys
from pathlib import Path

REPO = "https://github.com/tvh0ng/neural-pde-solvers.git"
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs").is_dir()), None)
if ROOT is None:
    subprocess.check_call(["git", "clone", "-q", REPO, "neural-pde-solvers"])
    ROOT = Path("neural-pde-solvers").resolve()
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[surrogate]"]
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from neural_pde_solvers import setup_logging
from neural_pde_solvers.surrogate import ExperimentConfig, run_experiment
from neural_pde_solvers.surrogate.plotting import plot_sample_predictions, plot_training_curves
from neural_pde_solvers.utils import deep_update, load_yaml

setup_logging()
print(f"TensorFlow {tf.__version__} | GPUs: {tf.config.list_physical_devices('GPU')}")

## 1. Choose a model and the data

Set `DATA_PATH` to your CSV (header-less; see `data/README.md`). On Colab, mount Google
Drive and point to the file there. If no file is found, a synthetic dataset with the same
layout is generated so the notebook still runs end to end.

In [ ]:
MODEL = "lstm"  # ann | cnn1d | rnn | gru | lstm | cnn_gru
DATA_PATH = None  # e.g. "/content/drive/MyDrive/Research/Project Man/Data1/Out_ML_cm.csv"
QUICK = os.environ.get("SURROGATE_QUICK") == "1"

# from google.colab import drive; drive.mount("/content/drive")   # <- uncomment on Colab

overrides = {"output_dir": f"results/surrogate/{MODEL}", "data": {}, "training": {}}
if DATA_PATH and Path(DATA_PATH).exists():
    overrides["data"]["path"] = DATA_PATH
else:
    sys.path.insert(0, str(ROOT / "scripts"))
    from make_synthetic_data import make_table

    synthetic = ROOT / "data" / "synthetic.csv"
    synthetic.parent.mkdir(exist_ok=True)
    np.savetxt(synthetic, make_table(3000), delimiter=",", fmt="%.8g")
    overrides["data"].update(path=str(synthetic), n_samples=3000)
    print(f"Using synthetic data: {synthetic}")
if QUICK:
    overrides["training"]["epochs"] = 3

cfg = ExperimentConfig.from_dict(
    deep_update(load_yaml(ROOT / f"configs/surrogate/{MODEL}.yaml"), overrides)
)
cfg

## 2. Train

In [ ]:
cfg.training.verbose = 2
result = run_experiment(cfg)
result.model.summary()

## 3. Evaluate

In [ ]:
pd.DataFrame(
    {
        (split, space): vals
        for split in ("train", "val", "test")
        if split in result.metrics
        for space, vals in result.metrics[split].items()
    }
).T

In [ ]:
fig = plot_training_curves(result.history)
fig.savefig(cfg.output_dir + "/training_curves.png", dpi=150)
plt.show()

In [ ]:
from neural_pde_solvers.surrogate import reshape_inputs

s = result.splits
pred = result.model.predict(reshape_inputs(s.x_test, result.layout), verbose=0)
fig = plot_sample_predictions(
    s.scaler_y.inverse_transform(s.y_test), s.scaler_y.inverse_transform(pred), n=5
)
fig.savefig(cfg.output_dir + "/test_predictions.png", dpi=150)
plt.show()

## 4. Reuse the trained model

```python
from neural_pde_solvers.surrogate import load_trained, reshape_inputs
model, scalers, cfg = load_trained("results/surrogate/lstm")
y = scalers["y"].inverse_transform(
        model.predict(reshape_inputs(scalers["x"].transform(x_raw), "sequence")))
```

or, inside this session, simply `result.predict(x_raw)`.